In [0]:
#notebook de exploração

import pandas as pd

# 1) Carrega os dados brutos (SIM - Mortalidade Geral 2023)
url = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/DO23OPEN.csv"
df = pd.read_csv(url, sep=";", encoding="ISO-8859-1", low_memory=False)

# 2) Filtra só óbitos não-fetais (decisão da Fase 2)
df = df[df["TIPOBITO"] == 2].copy()

# 3) Decodifica IDADE -> idade em anos
def decode_idade_anos(idade):
    if pd.isna(idade):
        return None
    idade = int(idade)
    unidade, valor = divmod(idade, 100)
    if unidade == 4:
        return valor
    elif unidade == 5:
        return 100 + valor
    elif unidade in (0, 1, 2, 3):
        return 0  # menor de 1 ano
    else:
        return None  # ignorado

df["IDADE_ANOS"] = df["IDADE"].apply(decode_idade_anos)

# 4) Faixa etária (padrão saúde pública)
def faixa_etaria(idade_anos):
    if idade_anos is None:
        return "Ignorada"
    if idade_anos <= 14:
        return "0-14"
    elif idade_anos <= 29:
        return "15-29"
    elif idade_anos <= 44:
        return "30-44"
    elif idade_anos <= 59:
        return "45-59"
    elif idade_anos <= 74:
        return "60-74"
    else:
        return "75+"

df["FAIXA_ETARIA"] = df["IDADE_ANOS"].apply(faixa_etaria)

# 5) Classificação por tema (CIRCOBITO como fonte primária, CID-10 como fallback)
def classifica_tema(causa, circ):
    if pd.isna(causa):
        return "Não classificado"
    letra = causa[0]
    if letra not in ("V", "W", "X", "Y"):
        return "Saúde"
    if circ == 1:
        return "Acidente"
    elif circ == 2:
        return "Suicídio"
    elif circ == 3:
        return "Homicídio"
    elif circ == 4:
        return "Outra causa externa"
    try:
        num = int(causa[1:3])
    except ValueError:
        return "Não classificado"
    if letra == "X" and 60 <= num <= 84:
        return "Suicídio"
    if (letra == "X" and 85 <= num <= 99) or (letra == "Y" and 0 <= num <= 9):
        return "Homicídio"
    if letra == "V" or letra == "W" or (letra == "X" and num <= 59):
        return "Acidente"
    return "Não classificado"

df["TEMA"] = df.apply(lambda r: classifica_tema(r["CAUSABAS"], r["CIRCOBITO"]), axis=1)

# 6) Subclassificação de "Saúde" por capítulo do CID-10
def classifica_saude(causa):
    if pd.isna(causa):
        return "Não classificado"
    letra = causa[0]
    try:
        num = int(causa[1:3])
    except ValueError:
        return "Não classificado"
    if letra == "U" and num == 7:
        return "COVID-19"
    if letra in ("A", "B"):
        return "Doenças infecciosas e parasitárias"
    if letra == "C" or (letra == "D" and num <= 48):
        return "Neoplasias (câncer)"
    if letra == "E":
        return "Doenças endócrinas, nutricionais e metabólicas"
    if letra == "F":
        return "Transtornos mentais e comportamentais"
    if letra == "G":
        return "Doenças do sistema nervoso"
    if letra == "I":
        return "Doenças do aparelho circulatório"
    if letra == "J":
        return "Doenças do aparelho respiratório"
    if letra == "K":
        return "Doenças do aparelho digestivo"
    if letra in ("P", "Q"):
        return "Causas perinatais e malformações congênitas"
    if letra == "R":
        return "Causas mal definidas"
    return "Outras doenças"

df["SUBTEMA"] = None
mask_saude = df["TEMA"] == "Saúde"
df.loc[mask_saude, "SUBTEMA"] = df.loc[mask_saude, "CAUSABAS"].apply(classifica_saude)

# 7) Rótulos de RACACOR e SEXO
racacor_map = {1.0: "Branca", 2.0: "Preta", 3.0: "Amarela", 4.0: "Parda", 5.0: "Indígena"}
df["RACACOR_DESC"] = df["RACACOR"].map(racacor_map).fillna("Ignorada")

sexo_map = {1: "Masculino", 2: "Feminino"}
df["SEXO_DESC"] = df["SEXO"].map(sexo_map).fillna("Ignorado")

print("Linhas carregadas e transformadas:", len(df))
df[["DTOBITO", "SEXO_DESC", "FAIXA_ETARIA", "RACACOR_DESC", "CODMUNRES", "TEMA", "SUBTEMA"]].head(10)